# Inverse elliptic-interface problem with Wavelet-PINN
Recover the unknown **interface shape** (and/or contrast) from sparse, noisy interior measurements of $u$.
Strategy: outer optimisation over the few physical parameters $\times$ inner AD-free least-squares forward solve.
Headline: interface-shape recovery, which classical FEM/IIM solvers do not provide natively.

In [ ]:
from config import *
from InverseProblem import *
from Wfamily import *
from Model import *

In [ ]:
# --- ground truth (flower) + synthetic sparse noisy measurements ---
r0_t, c2_t, c3_t, rho = 0.5, 0.0, 0.30, 10.0   # true 3-lobe flower
N, noise = 60, 1e-3
uf = truth_field(r0_t, c2_t, c3_t, rho); ub = uf(x_bc, y_bc)
xd = np.random.rand(N)*2-1; yd = np.random.rand(N)*2-1
keep = (np.abs(xd) < 0.97) & (np.abs(yd) < 0.97); xd, yd = xd[keep], yd[keep]
ud = uf(xd, yd)*(1 + noise*np.random.randn(len(xd)))
Pd, _, _, _ = W_mats(xd, yd)
print(f'{len(xd)} sparse interior measurements, noise={noise}')

In [ ]:
# --- inverse solve: minimise interior data misfit over geometry (r0, c3) ---
def J(p):
    r0, c3 = p
    if r0 < 0.25 or r0 > 0.8 or abs(c3) > 0.6: return 1e3
    phg = phi(XS, YS, r0, 0.0, c3); phid = phi(xd, yd, r0, 0.0, c3)
    cM, cP, bM, bP = forward(rho, phg, interface(r0, 0.0, c3), ub)
    pr = np.where(phid < 0, predict_at(cM, bM, Pd), predict_at(cP, bP, Pd))
    return np.sum((pr - ud)**2)
res = minimize(J, [0.4, 0.1], method='Nelder-Mead', options=dict(xatol=1e-4, fatol=1e-12, maxiter=400))
r0_h, c3_h = res.x
print(f'recovered (r0,c3)=({r0_h:.4f},{c3_h:.4f})  true=({r0_t:.2f},{c3_t:.2f})')

In [ ]:
# --- plot: true vs recovered interface + the sparse data, save sol.png ---
th = np.linspace(0, 2*np.pi, 400)
rt = radius_at(r0_t, c2_t, c3_t, th); rh = radius_at(r0_h, 0.0, c3_h, th)
fig, ax = plt.subplots(figsize=(5.2, 5))
ax.plot(rt*np.cos(th), rt*np.sin(th), 'k-', lw=2.5, label='true interface')
ax.plot(rh*np.cos(th), rh*np.sin(th), '--', color='tab:green', lw=2, label='recovered')
ax.scatter(xd, yd, s=14, c='tab:red', alpha=.7, label=f'{len(xd)} sparse data pts')
ax.set_aspect('equal'); ax.set_xlim(-1,1); ax.set_ylim(-1,1)
ax.set_title(f'interface SHAPE recovery\ntrue(r0,c3)=(0.50,0.30) -> ({r0_h:.3f},{c3_h:.3f})'); ax.legend(loc='upper right', fontsize=8)
plt.tight_layout(); plt.savefig('sol.png', dpi=130); plt.show()